# Stage12: Stakeholder Delivery

This notebook packages the fixed Stage10/11 outputs into a decision-oriented written report. It does not fit or retune a model.

In [1]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if not (ROOT / "src" / "reporting.py").is_file():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "project" / "src" / "reporting.py").is_file():
            ROOT = candidate / "project"
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.reporting import build_stakeholder_report

processed = ROOT / "data" / "processed"
reports = ROOT / "reports"
timestamp = "20260907-143336"


## Build the stakeholder-ready report

The report makes the decision boundary explicit: the alert supports human review, not automatic hedge execution. It includes uncertainty, an alternate cutoff scenario, and the lower-volatility subgroup risk.

In [2]:
report_path = build_stakeholder_report(
    processed / f"model_test_metrics_{timestamp}.csv",
    processed / f"bootstrap_pr_auc_{timestamp}.csv",
    processed / f"evaluation_scenarios_{timestamp}.csv",
    processed / f"evaluation_subgroups_{timestamp}.csv",
    f"spy_evaluation_{timestamp}.png",
    reports / "spy_risk_alert_stakeholder_report.md",
)
assert report_path.is_file()
print(f"Wrote: {report_path}")
print(report_path.read_text(encoding="utf-8")[:1200])

Wrote: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng/project/reports/spy_risk_alert_stakeholder_report.md
# SPY Next-Day High-Volatility Risk Alert
## Stakeholder Delivery - Portfolio Risk Management

### Executive summary

**Recommendation: use the baseline only as an end-of-day manual review trigger; do not automate a hedge or trade from this signal.**

- On the future-like test period, the alert captured 48.6% of high-volatility days (17 of 35), while creating alerts on 14.0% of sessions.
- The model has limited ranking signal (PR-AUC 0.293; 600-resample interval 0.179 to 0.457), so the estimated value is uncertain.
- Reliability is uneven: recall was 0.0% in the lower-volatility half of the test sample. This is a material reason not to use the score for automatic action.

### Decision context

The portfolio risk manager receives this output after the U.S. close and before the next market open. A high alert should trigger an analyst review of exposur